In [ ]:
import os
import pandas as pd

path_luz_processed = 'data/processed/Energia electrica 2025 ambos medidores.xlsx'

def procesar_hoja_medidores(sheet_name):
    """Procesa una pestaña anual aislando y renombrando las columnas de ambos medidores."""
    df_raw = pd.read_excel(path_luz_processed, sheet_name=sheet_name, header=None)
    
    # 1. Los datos reales de los meses arrancan en la fila 4
    df_datos = df_raw.iloc[4:, :].copy()
    
    # 2. Mapeo de columnas según la estructura de los medidores
    df_datos.rename(columns={
        1: 'Mes',
        2: 'kWh_Medidor_Produccion',       # Columna 2: Medidor 4156946 (Naves)
        3: 'Costo_ARS_Produccion',         # Columna 3: $ del medidor de producción
        7: 'Produccion_lts_kg',            # Columna 7: lts- kg (Volumen físico)
        11: 'kWh_Medidor_Servicios',       # Columna 11: kW-h del segundo medidor (Servicios)
        12: 'Costo_ARS_Servicios'          # Columna 12: $ del segundo medidor
    }, inplace=True)
    
    # Limpieza y normalización de la columna Mes
    df_datos['Mes'] = df_datos['Mes'].astype(str).str.lower().str.strip()
    meses_validos = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
    df_datos = df_datos[df_datos['Mes'].isin(meses_validos)].copy()
    
    # Seleccionamos solo las columnas analíticas que nos interesan para el modelo
    cols_finales = ['Mes', 'kWh_Medidor_Produccion', 'kWh_Medidor_Servicios', 'Produccion_lts_kg']
    cols_existentes = [c for c in cols_finales if c in df_datos.columns]
    df_final = df_datos[cols_existentes].copy()
    
    # 5. Forzamos la conversión a tipos numéricos puros (float64)
    for col in df_final.columns:
        if col != 'Mes':
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
            
    return df_final.reset_index(drop=True)

# Control de existencia antes de ejecutar
if not os.path.exists(path_luz_processed):
    print(f"No se encontró el archivo en la ruta relativa: {path_luz_processed}")
    print(f"Asegurate de estar parado en la raíz de 'Politecnico' al correr el notebook.")
else:
    # Ejecutamos el procesamiento para el año 2025
    df_luz_2025_limpio = procesar_hoja_medidores('datos25')

    print("=========================================================")
    output_cols = df_luz_2025_limpio.columns.tolist()
    print(f"✔️ Matriz unificada con éxito. Columnas finales: {output_cols}")
    print("=========================================================\n")
    print(df_luz_2025_limpio.head(6))

    # 6. Exportación directa a la carpeta processed como CSV limpio
    ruta_salida_csv = 'data/processed/consumo_luz_medidores_2025.csv'
    df_luz_2025_limpio.to_csv(ruta_salida_csv, index=False)

✔️ Matriz unificada con éxito. Columnas finales: ['Mes', 'kWh_Medidor_Produccion', 'kWh_Medidor_Servicios', 'Produccion_lts_kg']

       Mes  kWh_Medidor_Produccion  kWh_Medidor_Servicios  Produccion_lts_kg
0    enero                148340.0                    NaN             365645
1  febrero                161773.0                    NaN             419403
2    marzo                151287.0                    NaN             422762
3    abril                213995.0                    NaN            1518390
4     mayo                219258.4                    NaN            2413571
5    junio                256315.0                    NaN            2436084

💾 ¡Tabla hija guardada en limpio en: 'data/processed/consumo_luz_medidores_2025.csv'!


In [8]:
import os
import pandas as pd

# 1. Fijamos la ruta absoluta apuntando a la carpeta PROCESSED como me indicaste
path_luz_processed = 'data/processed/Energia electrica 2025 ambos medidores.xlsx'

# Control de existencia para blindar el script
if not os.path.exists(path_luz_processed):
    print(f"❌ No se encontró el archivo en 'data/processed/'. Verificá el nombre exacto.")
else:
    print(f"🎯 Archivo detectado con éxito en PROCESSED.")

    # 2. Cargamos las pestañas analíticas para revisar la estructura de los medidores
    # Probamos con 'datos25' que es donde tenías los datos planchados
    df_raw = pd.read_excel(path_luz_processed, sheet_name='datos25', header=None)
    
    print("\n--- ESTRUCTURA ENCONTRADA (Primeras 8 filas) ---")
    print(df_raw.head(8))

🎯 Archivo detectado con éxito en PROCESSED.

--- ESTRUCTURA ENCONTRADA (Primeras 8 filas) ---
   0                       1        2                               3   \
0 NaN                     NaN      NaN                             NaN   
1 NaN  ENERGIA ELECTRICA 2025      NaN                             NaN   
2 NaN                medidor   4156946  A300-400-500-600-700-1000-1100   
3 NaN                     NaN     Kw-h                               $   
4 NaN                   enero   148340                        30891849   
5 NaN                 febrero   161773                        35294347   
6 NaN                   marzo   151287                        36118133   
7 NaN                   abril   213995                        44270737   

        4             5                                      6        7   \
0      NaN           NaN                                    NaN      NaN   
1      NaN           NaN                                    NaN      NaN   
2      NaN 

In [14]:
import os
import pandas as pd

# 1. Definimos las rutas relativas reales basándonos en tu lista de PROCESSED
path_medidor1_2026 = 'data/processed/Energia electrica 2026 medidor 1.xlsx'
path_medidor2_2026 = 'data/processed/Energia electrica 2026 medidor 2.xlsx'

# Verificación rápida de que los archivos están en su lugar
if not os.path.exists(path_medidor1_2026) or not os.path.exists(path_medidor2_2026):
    print("❌ Alerta: Alguno de los archivos de 2026 no se encuentra en 'data/processed/'.")
    print("Revisá que las mayúsculas, minúsculas y espacios coincidan exactamente.")
else:
    print("🎯 Ambos archivos de Excel para 2026 detectados con éxito.\n")

    # 2. Leemos la primera pestaña de cada archivo de manera cruda (primeras filas)
    # Usamos header=None para inspeccionar el ruido y ver dónde arrancan los meses
    df_m1_raw = pd.read_excel(path_medidor1_2026, sheet_name=0, header=None)
    df_m2_raw = pd.read_excel(path_medidor2_2026, sheet_name=0, header=None)
    
    print("=== 🔬 ESTRUCTURA DETECTADA EN MEDIDOR 1 (Producción) ===")
    print(df_m1_raw.head(6))
    print("\n" + "="*50 + "\n")
    
    print("=== 🔬 ESTRUCTURA DETECTADA EN MEDIDOR 2 (Servicios) ===")
    print(df_m2_raw.head(6))

🎯 Ambos archivos de Excel para 2026 detectados con éxito.

=== 🔬 ESTRUCTURA DETECTADA EN MEDIDOR 1 (Producción) ===
   0       1                  2    3           4    5           6    7   \
0 NaN     NaN                NaN  NaN         NaN  NaN         NaN  NaN   
1 NaN     NaN  ENERGIA ELECTRICA  NaN         NaN  NaN         NaN  NaN   
2 NaN     NaN       COMPARATIVA   NaN         NaN  NaN         NaN  NaN   
3 NaN     NaN                NaN  NaN         NaN  NaN         NaN  NaN   
4 NaN     NaN             U$S/KW    %  U$S /lt-kg    %  KW / lt-kg    %   
5 NaN  2023.0           0.098001  NaN    0.017021  NaN    0.165414  NaN   

                           8   9       10        11          12          13  
0                         NaN NaN     NaN       NaN         NaN         NaN  
1                         NaN NaN     NaN       NaN         NaN         NaN  
2                         NaN NaN     NaN       NaN         NaN         NaN  
3                         NaN NaN     NaN   EN